# Build

In [17]:
"""
QuTiP notebook-friendly ED time evolution for the Hamiltonian

    H = sum_{j=0}^{N-2} Jz * σ^z_j σ^z_{j+1}
      + sum_{i<j} (Jx / |j-i|^alpha) * σ^x_i σ^x_j

Utilities compute observables and save them to CSV.
"""

from __future__ import annotations
from typing import List, Optional, Tuple, Dict
import os

import numpy as np
import pandas as pd
import qutip as qt

def op_on_site(op: qt.Qobj, N: int, site: int) -> qt.Qobj:
    """Return I ⊗ ... ⊗ op ⊗ ... ⊗ I with op acting on `site`."""
    if N <= 0:
        raise ValueError("N must be positive.")
    if not (0 <= site < N):
        raise ValueError(f"site={site} out of range for N={N}.")
    ops = [qt.qeye(2)] * N
    ops[site] = op
    return qt.tensor(ops)

def two_site_op(opA: qt.Qobj, opB: qt.Qobj, N: int, i: int, j: int) -> qt.Qobj:
    """Return I ⊗ ... ⊗ opA ⊗ ... ⊗ opB ⊗ ... ⊗ I with opA on site i and opB on site j."""
    if N <= 0:
        raise ValueError("N must be positive.")
    if i == j:
        raise ValueError("i and j must be different sites.")
    if not (0 <= i < N and 0 <= j < N):
        raise ValueError(f"Sites (i={i}, j={j}) out of range for N={N}.")
    ops = [qt.qeye(2)] * N
    ops[i] = opA
    ops[j] = opB
    return qt.tensor(ops)

# -----------------------------
# Hamiltonian
# -----------------------------

def kac_factor(N: int, alpha: float) -> float:
    """
    Returns Kac factor K such that Jlong = Jz / K.
    Matches: K = sum_{i<j} 1/r^alpha divided by (N-1).
    """
    r = np.arange(1, N, dtype=float)
    # sum_{i<j} 1/r^alpha = sum_{r=1}^{N-1} (N-r)/r^alpha
    sumV = np.sum((N - r) / (r ** alpha))
    return sumV / (N - 1)

def build_H_B_Jz_Jlong(N: int, Bx: float, Jx: float, Jz: float, alpha: float, kac: bool = True) -> qt.Qobj:
    """
    Construct the long-range Ising-type Hamiltonian:

        H = Jx * sum_{j=0}^{N-2} σ^x_j σ^x_{j+1}
          + sum_{i<j} (Jz / |i - j|^alpha) σ^z_i σ^z_j

    where:
        - Jx controls nearest-neighbor interactions in the x-basis
        - Jz controls long-range interactions in the z-basis
        - alpha sets the decay of the long-range coupling:
            alpha = 0   → all-to-all (uniform coupling)
            alpha → ∞   → effectively nearest-neighbor
        - kac=True applies Kac normalization to keep energy extensive:

    Notes:
        - Open boundary conditions are used
        - two_site_op(opA, opB, N, i, j) places operators at sites i and j

    Returns:
        qt.Qobj representing the Hamiltonian
    """
    if N < 2:
        raise ValueError("N must be >= 2.")
    if alpha < 0:
        raise ValueError("alpha must be >= 0.")

    sx, sz = qt.sigmax(), qt.sigmaz()
    H = 0

    if kac:
        K = kac_factor(N, alpha)
        Jz = Jz / K  # rescaled long-range prefactor
    # ------------------------------------------------------------------
    # On-site field term:
    #
    #   H_B = Bx * sum_j σ^x_j
    # ------------------------------------------------------------------
    for j in range(N):
        H += Bx * op_on_site(sx, N, j)

    # # Nearest-neighbor xx
    for j in range(N - 1):
        H += Jx * two_site_op(sx, sx, N, j, j + 1)

    # ------------------------------------------------------------------
    # Construct a localized zz interaction Hamiltonian centered on a single site:

    #     H = Jz * (σ^z_c σ^z_{c-1} + σ^z_c σ^z_{c+1})

    # Description:
    #     - Only a single “central” site c interacts with its immediate neighbors.
    #     - No other zz couplings are present in the system.
    #     - Effectively creates a local impurity or interaction hub.
    # ------------------------------------------------------------------

    # center = 1  # Python index for paper site 2

    # if center - 1 >= 0:
    #     H += Jz * two_site_op(sz, sz, N, center, center - 1)
    # if center + 1 < N:
    #     H += Jz * two_site_op(sz, sz, N, center, center + 1)

    # zz term
    if alpha >= 100:
        for j in range(N - 1):
            H += Jz * two_site_op(sz, sz, N, j, j + 1)
    else:
        for i in range(N):
            for j in range(i + 1, N):
                r = j - i
                H += (Jz / (r ** alpha)) * two_site_op(sz, sz, N, i, j)

    return H

# -----------------------------
# Initial states
# -----------------------------

def superpose(psi1: qt.Qobj, psi2: qt.Qobj, normalize: bool = True) -> qt.Qobj:
    """
    Return the superposition psi1 + psi2.

    If normalize=True, returns a normalized state.
    """
    if not (psi1.isket and psi2.isket):
        raise ValueError("Both inputs must be kets.")
    if psi1.shape != psi2.shape:
        raise ValueError("States have different Hilbert-space dimensions.")
    if psi1.dims != psi2.dims:
        raise ValueError("States have incompatible dims; fix before superposing.")
    psi = psi1 + psi2
    return psi.unit() if normalize else psi


def product_state_from_bitstring(bitstring: str) -> qt.Qobj:
    """
    Construct a product state |b1 b2 ... bN⟩ from a bitstring of '0' and '1'.
    |0> ≡ spin-up, |1> ≡ spin-down (computational basis)
    """
    s = bitstring.strip().replace(" ", "")
    if not s or any(c not in "01" for c in s):
        raise ValueError("bitstring must be a non-empty string of 0/1 characters.")
    kets = [qt.basis(2, int(c)) for c in s]
    return qt.tensor(kets).unit()

def product_state_angles(thetas: np.ndarray, phis: np.ndarray) -> qt.Qobj:
    """
    Construct a product state from Bloch-sphere angles.

    Each site is:
        |ψ(θ, φ)⟩ = cos(θ/2)|0⟩ + e^{iφ} sin(θ/2)|1⟩

    Parameters
    ----------
    thetas : array_like
        Polar angles θ ∈ [0, π] for each site
    phis : array_like
        Azimuthal angles φ ∈ [0, 2π) for each site

    Returns
    -------
    qt.Qobj
        Tensor-product state ⊗_i |ψ(θ_i, φ_i)⟩
    """
    thetas = np.asarray(thetas, dtype=float)
    phis = np.asarray(phis, dtype=float)
    if thetas.shape != phis.shape:
        raise ValueError("thetas and phis must have the same shape.")
    N = thetas.size
    if N == 0:
        raise ValueError("Need at least one site.")

    kets = []
    for th, ph in zip(thetas, phis):
        ket = np.cos(th / 2) * qt.basis(2, 0) + np.exp(1j * ph) * np.sin(th / 2) * qt.basis(2, 1)
        kets.append(ket)
    return qt.tensor(kets).unit()


def product_state_x_pm(pm: str) -> qt.Qobj:
    """
    Construct a product state in the σ^x basis using '+' / '-' notation.

    '+' → |+⟩ = (|0⟩ + |1⟩)/√2   (φ = 0)
    '-' → |−⟩ = (|0⟩ − |1⟩)/√2   (φ = π)

    Parameters
    ----------
    pm : str
        String of '+' and '-' specifying each site

    Returns
    -------
    qt.Qobj
        Tensor-product state in the σ^x basis
    """
    pm = pm.strip().replace(" ", "")
    if not pm or any(c not in "+-" for c in pm):
        raise ValueError("pm must be a non-empty string of '+'/'-'.")
    N = len(pm)
    thetas = np.full(N, np.pi / 2)
    phis = np.array([0.0 if c == "+" else np.pi for c in pm], dtype=float)
    return product_state_angles(thetas, phis)

# -----------------------------
# Dynamics
# -----------------------------

def evolve_state(H: qt.Qobj, psi0: qt.Qobj, times: np.ndarray) -> List[qt.Qobj]:
    """
    Evolve an initial state under a time-independent Hamiltonian using QuTiP's `mesolve`.

    This solves the von Neumann equation:
        dρ/dt = -i [H, ρ]

    Notes
    -----
    - If `psi0` is a ket, QuTiP internally converts it to a density matrix ρ = |ψ⟩⟨ψ|.
    - With no collapse operators (c_ops=[]), the evolution is unitary and equivalent
      to `sesolve`, but the states are returned as density matrices instead of kets.
    - Returned objects are density matrices even for pure-state evolution.

    Parameters
    ----------
    H : qt.Qobj
        Hamiltonian operator
    psi0 : qt.Qobj
        Initial state (ket or density matrix)
    times : array_like
        1D array of time points at which to evaluate the state

    Returns
    -------
    List[qt.Qobj]
        List of density matrices ρ(t) at each time in `times`
    """
    times = np.asarray(times, dtype=float)
    res = qt.mesolve(H, psi0, times, c_ops=[], e_ops=[])
    return res.states


# -----------------------------
# Observables
# -----------------------------

def magnetization_z(psi: qt.Qobj, N: int, sz_ops: Optional[List[qt.Qobj]] = None) -> np.ndarray:
    """
    Compute site-resolved magnetization ⟨σ^z_i⟩ for a given state.

    Parameters
    ----------
    psi : qt.Qobj
        State vector (ket)
    N : int
        Number of sites
    sz_ops : list of qt.Qobj, optional
        Precomputed σ^z operators for each site. If not provided,
        they are constructed internally.

    Returns
    -------
    np.ndarray
        Array of length N with ⟨σ^z_i⟩ at each site
    """
    if sz_ops is None:
        sz = qt.sigmaz()
        sz_ops = [op_on_site(sz, N, i) for i in range(N)]
    return np.array([qt.expect(sz_ops[i], psi).real for i in range(N)], dtype=float)


def magnetization_x(psi: qt.Qobj, N: int, sx_ops: Optional[List[qt.Qobj]] = None) -> np.ndarray:
    """
    Compute site-resolved magnetization ⟨σ^x_i⟩ for a given state.

    Parameters
    ----------
    psi : qt.Qobj
        State vector (ket)
    N : int
        Number of sites
    sx_ops : list of qt.Qobj, optional
        Precomputed σ^x operators for each site. If not provided,
        they are constructed internally.

    Returns
    -------
    np.ndarray
        Array of length N with ⟨σ^x_i⟩ at each site
    """
    if sx_ops is None:
        sx = qt.sigmax()
        sx_ops = [op_on_site(sx, N, i) for i in range(N)]
    return np.array([qt.expect(sx_ops[i], psi).real for i in range(N)], dtype=float)


def correlations_xx_connected(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    xx_ops: Optional[Dict[Tuple[int, int], qt.Qobj]] = None,
    sx_ops: Optional[List[qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    """
    Compute connected two-point correlators ⟨σ^x_i σ^x_j⟩_c.

    The connected correlator is defined as:
        ⟨σ^x_i σ^x_j⟩_c = ⟨σ^x_i σ^x_j⟩ - ⟨σ^x_i⟩⟨σ^x_j⟩

    For i = j, this reduces to:
        ⟨(σ^x_i)^2⟩ - ⟨σ^x_i⟩^2 = 1 - ⟨σ^x_i⟩^2

    Parameters
    ----------
    psi : qt.Qobj
        State vector (ket)
    N : int
        Number of sites
    pairs : list of (int, int)
        Site index pairs (i, j) for which to compute correlations
    xx_ops : dict, optional
        Precomputed σ^x_i σ^x_j operators keyed by (i, j)
    sx_ops : list of qt.Qobj, optional
        Precomputed σ^x_i operators

    Returns
    -------
    dict
        Mapping (i, j) → connected correlator value
    """
    sx = qt.sigmax()
    if sx_ops is None:
        sx_ops = [op_on_site(sx, N, i) for i in range(N)]

    mx = np.array([qt.expect(sx_ops[i], psi).real for i in range(N)], dtype=float)

    out: Dict[Tuple[int, int], float] = {}
    for (i, j) in pairs:
        if i == j:
            out[(i, j)] = float(1.0 - mx[i] * mx[i])
            continue

        if xx_ops is None:
            op = two_site_op(sx, sx, N, i, j)
            xx = qt.expect(op, psi).real
        else:
            xx = qt.expect(xx_ops[(i, j)], psi).real

        out[(i, j)] = float(xx - mx[i] * mx[j])

    return out

def correlations_zz_connected(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    zz_ops: Optional[Dict[Tuple[int, int], qt.Qobj]] = None,
    sz_ops: Optional[List[qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    """
    Compute connected two-point correlators ⟨σ^z_i σ^z_j⟩_c.

    Defined as:
        ⟨σ^z_i σ^z_j⟩_c = ⟨σ^z_i σ^z_j⟩ - ⟨σ^z_i⟩⟨σ^z_j⟩

    For i = j:
        ⟨(σ^z_i)^2⟩ - ⟨σ^z_i⟩^2 = 1 - ⟨σ^z_i⟩^2

    Parameters
    ----------
    psi : qt.Qobj
        State vector (ket)
    N : int
        Number of sites
    pairs : list of (int, int)
        Site pairs (i, j)
    zz_ops : dict, optional
        Precomputed σ^z_i σ^z_j operators
    sz_ops : list of qt.Qobj, optional
        Precomputed σ^z_i operators

    Returns
    -------
    dict
        Mapping (i, j) → connected correlator
    """
    sz = qt.sigmaz()
    if sz_ops is None:
        sz_ops = [op_on_site(sz, N, i) for i in range(N)]

    mz = np.array([qt.expect(sz_ops[i], psi).real for i in range(N)], dtype=float)

    out: Dict[Tuple[int, int], float] = {}
    for (i, j) in pairs:
        if i == j:
            out[(i, j)] = float(1.0 - mz[i] * mz[i])
            continue

        if zz_ops is None:
            op = two_site_op(sz, sz, N, i, j)
        else:
            op = zz_ops[(i, j)]

        zz = qt.expect(op, psi).real
        out[(i, j)] = float(zz - mz[i] * mz[j])

    return out


def concurrence_pair(psi: qt.Qobj, i: int, j: int) -> float:
    """
    Compute the two-qubit concurrence between sites i and j.

    The state is reduced to the two-site density matrix ρ_ij via partial trace,
    and the concurrence is computed using QuTiP's `concurrence`.

    Parameters
    ----------
    psi : qt.Qobj
        Pure state (ket) of the full system
    i, j : int
        Site indices

    Returns
    -------
    float
        Concurrence C ∈ [0, 1], measuring bipartite entanglement
    """
    if i == j:
        return 0.0
    rho_ij = qt.ptrace(psi, [i, j])
    return float(qt.concurrence(rho_ij))


# def compute_metrics_x_basis(N: int, states: List[qt.Qobj], psi0: qt.Qobj):
#     """
#     Computes Leakage, Fidelity, and IPR specifically in the Sigma-X basis.
#     """
#     # 1. Define B1 Sector in X-basis
#     # B1 = states with exactly one '-' and (N-1) '+'
#     b1_x_basis = []
#     for i in range(N):
#         # Create a string like "+++-+ +"
#         x_string = ["+"] * N
#         x_string[i] = "-"
#         b1_x_basis.append(product_state_x_pm("".join(x_string)))
    
#     # Projector onto the B1 sector (X-basis)
#     P_B1_x = sum([state * state.dag() for state in b1_x_basis])
    
#     # 2. Define the full X-basis for IPR
#     # To compute IPR in X, we need to transform the state to the X-basis 
#     # or project onto all 2^N X-basis combinations.
#     # A more efficient way: Rotate the state to Z-basis using the Hadamard transform
#     # and then compute standard IPR.
    
#     # Single-qubit rotation: H maps |0> -> |+> and |1> -> |->
#     H_gate = (qt.sigmax() + qt.sigmaz()) / np.sqrt(2)
#     Full_H_gate = qt.tensor([H_gate] * N)
    
#     leakage = []
#     fidelity = []
#     ipr = []
    
#     for psi in states:
#         # Leakage: 1 - Prob(being in B1 sector of X)
#         prob_in_b1 = qt.expect(P_B1_x, psi)
#         leakage.append(max(0, 1.0 - prob_in_b1)) # max to handle tiny precision errors
        
#         # Fidelity: |<psi0|psi(t)>|^2 (Basis independent)
#         f = np.abs(psi0.dag() * psi)**2
#         fidelity.append(f.flatten()[0])
        
#         # IPR in X-basis:
#         # Rotate psi back to the Z-basis to treat |+>/|-> as |0>/|1>
#         psi_in_x_coords = Full_H_gate.dag() * psi
#         probs = np.abs(psi_in_x_coords.full())**2
#         ipr.append(np.sum(probs**2))
        
#     return np.array(leakage), np.array(fidelity), np.array(ipr)

# -----------------------------
# Simulation + CSV output
# -----------------------------

def run_and_save_csv(
    N: int,
    Bx: float,
    Jx: float,
    Jz: float,
    alpha: float,
    times: np.ndarray,
    prefix: str = "data",
    outdir: str = "data_qutip",
    psi0: Optional[qt.Qobj] = None,
    bitstring: Optional[str] = None,
    pmstring: Optional[str] = None,
    superpose_spec: Optional[str] = None,
    corrxx_pairs: Optional[List[Tuple[int, int]]] = None,
    corrzz_pairs: Optional[List[Tuple[int, int]]] = None,
    conc_pairs: Optional[List[Tuple[int, int]]] = None,
    do_corrzz: bool = True,
    do_corrxx: bool = True,
    do_conc: bool = True,
    do_mx: bool = True,
    do_mz: bool = False,
    do_global_metrics: bool = True, # New flag for Leakage, Fidelity, IPR
    kac: bool = True,
) -> qt.Qobj:

    os.makedirs(outdir, exist_ok=True)
    times = np.asarray(times, dtype=float)

    # Build Hamiltonian
    # H = build_H_Bz_Jlong(N, Bz, Jx, alpha, kac=kac)
    # def build_H_B_Jz_Jlong(N: int, Bx: float, Jx: float, Jz: float, alpha: float, kac: bool = True) -> qt.Qobj:
    H = build_H_B_Jz_Jlong(N, Bx, Jx, Jz, alpha, kac=kac)

    # Choose initial state
    if psi0 is None:
        if superpose_spec is not None:
            spec = superpose_spec.strip().replace(" ", "")
            if "|" not in spec:
                raise ValueError("superpose_spec must look like '<spec1>|<spec2>'")
            s1, s2 = spec.split("|", 1)

            def build_from_spec(s: str) -> qt.Qobj:
                if set(s) <= set("01"):
                    if len(s) != N:
                        raise ValueError(f"bitstring length {len(s)} must equal N={N}.")
                    return product_state_from_bitstring(s)
                if set(s) <= set("+-"):
                    if len(s) != N:
                        raise ValueError(f"pmstring length {len(s)} must equal N={N}.")
                    return product_state_x_pm(s)
                raise ValueError("Each spec must be only '0/1' or only '+/-'.")

            psi0 = superpose(build_from_spec(s1), build_from_spec(s2), normalize=True)

        elif pmstring is not None:
            s = pmstring.strip().replace(" ", "")
            if len(s) != N:
                raise ValueError(f"pmstring length {len(s)} must equal N={N}.")
            psi0 = product_state_x_pm(s)

        elif bitstring is not None:
            s = bitstring.strip().replace(" ", "")
            if len(s) != N:
                raise ValueError(f"bitstring length {len(s)} must equal N={N}.")
            psi0 = product_state_from_bitstring(s)

        else:
            raise ValueError("Provide psi0, or bitstring, or pmstring, or superpose_spec.")

    # Enforce consistent dims (helps ptrace/concurrence)
    psi0 = qt.Qobj(psi0.full(), dims=[[2] * N, [1] * N]).unit()

    # Evolve
    states = evolve_state(H, psi0, times)

    # --- Setup for Global Metrics (X-basis) ---
    if do_global_metrics:
        # B1 Sector Projector in X-basis
        b1_x_basis = []
        for i in range(N):
            x_str = ["+"] * N
            x_str[i] = "-"
            b1_x_basis.append(product_state_x_pm("".join(x_str)))
        P_B1_x = sum([s * s.dag() for s in b1_x_basis])
        
        # Rotation for IPR in X-basis
        H_gate = (qt.sigmax() + qt.sigmaz()) / np.sqrt(2)
        Full_H_gate = qt.tensor([H_gate] * N)

    # Pairs + flags
    corrxx_pairs = corrxx_pairs or []
    conc_pairs = conc_pairs or []
    corrzz_pairs = corrzz_pairs or []

    do_corrzz = bool(do_corrzz and len(corrzz_pairs) > 0)
    do_corrxx = bool(do_corrxx and len(corrxx_pairs) > 0)
    do_conc = bool(do_conc and len(conc_pairs) > 0)

    # Cache 1-site ops
    sx, sz = qt.sigmax(), qt.sigmaz()
    sx_ops = [op_on_site(sx, N, i) for i in range(N)]
    sz_ops = [op_on_site(sz, N, i) for i in range(N)]

    # Cache xx ops for corr (skip self)
    xx_ops = None
    if do_corrxx:
        xx_ops = {(i, j): two_site_op(sx, sx, N, i, j) for (i, j) in corrxx_pairs if i != j}

    # Cache zz ops for corrzz (skip self)
    zz_ops = None
    if do_corrzz:
        zz_ops = {(i, j): two_site_op(sz, sz, N, i, j) for (i, j) in corrzz_pairs if i != j}


    

    # Rows
    mx_rows: List[List[float]] = []
    mz_rows: List[List[float]] = []
    corrxx_rows: List[List[float]] = []
    conc_rows: List[List[float]] = []
    corrzz_rows: List[List[float]] = []
    global_rows = [] # [t, leakage, fidelity, ipr]

    for t, psi in zip(times, states):
        if do_mx:
            mx = magnetization_x(psi, N, sx_ops=sx_ops)
            mx_rows.append([float(t)] + [float(x) for x in mx])

        if do_mz:
            mz = magnetization_z(psi, N, sz_ops=sz_ops)
            mz_rows.append([float(t)] + [float(z) for z in mz])

        if do_corrxx:
            corr = correlations_xx_connected(psi, N, corrxx_pairs, xx_ops=xx_ops, sx_ops=sx_ops)
            corrxx_rows.append([float(t)] + [float(corr[p]) for p in corrxx_pairs])

        if do_conc:
            conc = [concurrence_pair(psi, i, j) for (i, j) in conc_pairs]
            conc_rows.append([float(t)] + [float(x) for x in conc])
        
        if do_corrzz:
            corrzz = correlations_zz_connected(psi, N, corrzz_pairs, zz_ops=zz_ops, sz_ops=sz_ops)
            corrzz_rows.append([float(t)] + [float(corrzz[p]) for p in corrzz_pairs])

        if do_global_metrics:
            # Leakage
            leak = 1.0 - qt.expect(P_B1_x, psi)
            # Fidelity
            fid = np.abs(psi0.dag() * psi)**2
            # IPR (rotated to X-basis)
            psi_x = Full_H_gate.dag() * psi
            ipr = np.sum(np.abs(psi_x.full())**4)
            global_rows.append([float(t), float(leak), float(fid), float(ipr)])

    # Save
    if do_mx:
        pd.DataFrame(mx_rows, columns=["t"] + [f"mx_{i}" for i in range(N)]).to_csv(
            os.path.join(outdir, f"{prefix}magx.csv"), index=False
        )

    if do_mz:
        pd.DataFrame(mz_rows, columns=["t"] + [f"mz_{i}" for i in range(N)]).to_csv(
            os.path.join(outdir, f"{prefix}magz.csv"), index=False
        )

    if do_corrxx:
        pd.DataFrame(corrxx_rows, columns=["t"] + [f"Cxx_{i}_{j}" for (i, j) in corrxx_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}corrxx.csv"), index=False
        )

    if do_conc:
        pd.DataFrame(conc_rows, columns=["t"] + [f"C_{i}_{j}" for (i, j) in conc_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}conc.csv"), index=False
        )
    
    if do_corrzz:
        pd.DataFrame(corrzz_rows, columns=["t"] + [f"Czz_{i}_{j}" for (i, j) in corrzz_pairs]).to_csv(
            os.path.join(outdir, f"{prefix}corrzz.csv"), index=False
        )

    if do_global_metrics:
        cols = ["t", "leakage_x", "fidelity", "ipr_x"]
        pd.DataFrame(global_rows, columns=cols).to_csv(
            os.path.join(outdir, f"{prefix}global_metrics.csv"), index=False)

    return H

# Run

In [36]:
N = 10
Jx = 1
Bx= 0.1
Jz=100
# print(f"{Jx*10:.0f}")
alpha = 0.5
i0=1
tmax=5
tInterval = tmax*10
times = np.linspace(0.0, tmax, tInterval+1)
outdir="ED_Data/Confinement"
prefix=f"Bx{Bx}Jz{Jz}"

# Example: only magnetization (comment out corr/conc cleanly via flags)
#bitstring="00101"
# pmstring="+-+++"
# superpose_spec="00100|00010"
# superpose_spec="+-+-+|00101"
# Bitstring must have length N
H = run_and_save_csv(
    N=N, 
    Bx=Bx, 
    Jx=Jx, 
    Jz=Jz,
    alpha=alpha, 
    times=times,
    prefix=prefix,
    outdir=outdir,
    bitstring="1011111111",
    # pmstring="-+--------",
    # superpose_spec="-+------------|--+-----------",
    # superpose_spec="10111111111111|11011111111111",
    #pmstring="-+--------",
    corrxx_pairs=[(i0, j) for j in range(N)],     # includes self
    corrzz_pairs=[(i0, j) for j in range(N)], 
    conc_pairs=[(i0, j) for j in range(N)],
    #conc_pairs=[(i, j) for i in range(N) for j in range(N)],
do_corrxx=True, do_conc=False, do_corrzz=True, do_mx=True, do_mz=True,do_global_metrics=True,
kac=False,)
print(f"Saved CSVs {Jz*10:.0f} to {outdir}/")

Saved CSVs 1000 to ED_Data/Confinement/


# Debug

## Measures

In [ ]:
def magnetization_z(psi: qt.Qobj, N: int, sz_ops: Optional[List[qt.Qobj]] = None) -> np.ndarray:
    """
    Compute site-resolved magnetization ⟨σ^z_i⟩ for a given state.

    Parameters
    ----------
    psi : qt.Qobj
        State vector (ket)
    N : int
        Number of sites
    sz_ops : list of qt.Qobj, optional
        Precomputed σ^z operators for each site. If not provided,
        they are constructed internally.

    Returns
    -------
    np.ndarray
        Array of length N with ⟨σ^z_i⟩ at each site
    """
    if sz_ops is None:
        sz = qt.sigmaz()
        sz_ops = [op_on_site(sz, N, i) for i in range(N)]
    return np.array([qt.expect(sz_ops[i], psi).real for i in range(N)], dtype=float)

All up (or down) along z

In [ ]:
up=qt.basis(2, 0)
# up=qt.basis(2, 1) #down
psi = qt.tensor([up for _ in range(2)])
print(magnetization_z(psi, 2))

[-1. -1.]


All up (or down) along x

In [24]:
up = qt.basis(2, 0)
down = qt.basis(2, 1)

plus = (up + down).unit()
minus = (up - down).unit()

psi = qt.tensor([plus, minus])
print("magz =", magnetization_z(psi, 2))

magz = [0. 0.]


In [ ]:

def correlations_zz_connected(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    zz_ops: Optional[Dict[Tuple[int, int], qt.Qobj]] = None,
    sz_ops: Optional[List[qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    """
    Compute connected two-point correlators ⟨σ^z_i σ^z_j⟩_c.

    Defined as:
        ⟨σ^z_i σ^z_j⟩_c = ⟨σ^z_i σ^z_j⟩ - ⟨σ^z_i⟩⟨σ^z_j⟩

    For i = j:
        ⟨(σ^z_i)^2⟩ - ⟨σ^z_i⟩^2 = 1 - ⟨σ^z_i⟩^2
        
    Returns
    -------
    dict
        Mapping (i, j) → connected correlator
    """
    sz = qt.sigmaz()
    if sz_ops is None:
        sz_ops = [op_on_site(sz, N, i) for i in range(N)]

    mz = np.array([qt.expect(sz_ops[i], psi).real for i in range(N)], dtype=float)

    out: Dict[Tuple[int, int], float] = {}
    for (i, j) in pairs:
        if i == j:
            out[(i, j)] = float(1.0 - mz[i] * mz[i])
            continue

        if zz_ops is None:
            op = two_site_op(sz, sz, N, i, j)
        else:
            op = zz_ops[(i, j)]

        zz = qt.expect(op, psi).real
        out[(i, j)] = float(zz - mz[i] * mz[j])

    return out

In [29]:
up=qt.basis(2, 0)
down=qt.basis(2, 1) #down
psi = qt.tensor([up, down])
pairs=[(0,0), (0,1)]
print(correlations_zz_connected(psi, 2, pairs))

{(0, 0): 0.0, (0, 1): 0.0}


In [32]:
up = qt.basis(2, 0)
down = qt.basis(2, 1)

plus = (up + down).unit()
minus = (up - down).unit()

psi = qt.tensor([plus, plus])
pairs=[(0,0), (0,1), (1,0) , (1,1)]
print(correlations_zz_connected(psi, 2, pairs))

{(0, 0): 1.0, (0, 1): 0.0, (1, 0): 0.0, (1, 1): 1.0}


In [33]:
def correlations_xx_connected(
    psi: qt.Qobj,
    N: int,
    pairs: List[Tuple[int, int]],
    xx_ops: Optional[Dict[Tuple[int, int], qt.Qobj]] = None,
    sx_ops: Optional[List[qt.Qobj]] = None,
) -> Dict[Tuple[int, int], float]:
    """
    Compute connected two-point correlators ⟨σ^x_i σ^x_j⟩_c.

    The connected correlator is defined as:
        ⟨σ^x_i σ^x_j⟩_c = ⟨σ^x_i σ^x_j⟩ - ⟨σ^x_i⟩⟨σ^x_j⟩

    For i = j, this reduces to:
        ⟨(σ^x_i)^2⟩ - ⟨σ^x_i⟩^2 = 1 - ⟨σ^x_i⟩^2

    Parameters
    ----------
    psi : qt.Qobj
        State vector (ket)
    N : int
        Number of sites
    pairs : list of (int, int)
        Site index pairs (i, j) for which to compute correlations
    xx_ops : dict, optional
        Precomputed σ^x_i σ^x_j operators keyed by (i, j)
    sx_ops : list of qt.Qobj, optional
        Precomputed σ^x_i operators

    Returns
    -------
    dict
        Mapping (i, j) → connected correlator value
    """
    sx = qt.sigmax()
    if sx_ops is None:
        sx_ops = [op_on_site(sx, N, i) for i in range(N)]

    mx = np.array([qt.expect(sx_ops[i], psi).real for i in range(N)], dtype=float)

    out: Dict[Tuple[int, int], float] = {}
    for (i, j) in pairs:
        if i == j:
            out[(i, j)] = float(1.0 - mx[i] * mx[i])
            continue

        if xx_ops is None:
            op = two_site_op(sx, sx, N, i, j)
            xx = qt.expect(op, psi).real
        else:
            xx = qt.expect(xx_ops[(i, j)], psi).real

        out[(i, j)] = float(xx - mx[i] * mx[j])

    return out

In [34]:
up=qt.basis(2, 0)
down=qt.basis(2, 1) #down
psi = qt.tensor([up, down])
pairs=[(0,0), (0,1)]
print(correlations_xx_connected(psi, 2, pairs))

{(0, 0): 1.0, (0, 1): 0.0}


In [37]:
up = qt.basis(2, 0)
down = qt.basis(2, 1)

plus = (up + down).unit()
minus = (up - down).unit()

psi = qt.tensor([plus, minus])
pairs=[(0,0), (0,1)]
print(correlations_xx_connected(psi, 2, pairs))

{(0, 0): 8.881784197001252e-16, (0, 1): -4.440892098500626e-16}


## Debug global metrics

In [ ]:
def compute_leakage_and_fidelity(
    states: List[qt.Qobj],
    psi0: qt.Qobj,
    P_subspace: qt.Qobj,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute leakage and initial-state fidelity as functions of time.

    Definitions
    -----------
    Leakage:
        L(t) = 1 - Tr[P_subspace ρ(t)]

    Fidelity (with respect to initial pure state |ψ₀⟩):
        F(t) = ⟨ψ₀| ρ(t) |ψ₀⟩

    Notes
    -----
    - Works for both pure states (kets) and density matrices.
    - If states are kets, they are internally treated as ρ = |ψ⟩⟨ψ|.
    - `P_subspace` should be a projector onto the desired subspace (e.g., B1 sector).

    Parameters
    ----------
    states : list of qt.Qobj
        Time-evolved states (kets or density matrices)
    psi0 : qt.Qobj
        Initial pure state (ket)
    P_subspace : qt.Qobj
        Projector onto the subspace used to define leakage

    Returns
    -------
    leakage : np.ndarray
        Leakage values L(t)
    fidelity : np.ndarray
        Fidelity values F(t)
    """
    leakage = []
    fidelity = []

    for state in states:
        # Convert ket -> density matrix if needed
        if state.isket:
            rho = state * state.dag()
        else:
            rho = state

        # Leakage
        prob_in_subspace = qt.expect(P_subspace, rho)
        leakage.append(float(max(0.0, 1.0 - prob_in_subspace)))

        # Fidelity
        fid = qt.expect(psi0 * psi0.dag(), rho)
        fidelity.append(float(fid))

    return np.array(leakage), np.array(fidelity)